# MCP 등록·삭제 — tools 배열을 안 건드리고 캐시 지키기 (델타 고지 방식)

`cc_toolsearch_kv_cache.ipynb`의 후속편입니다. 앞 노트북은 "도구가 많을 때 tools 배열을 동결하는 법"을 다뤘고,
이번에는 **세션 중간에 도구 목록 자체가 바뀌는 상황** — MCP 서버 등록(연결) / 삭제(해제) — 을 다룹니다.

클로드코드의 답은 세 가지 장치입니다:
1. MCP 도구는 애초에 tools 배열에 넣지 않는다 (`isMcp: true` → 무조건 defer, `MCPTool.ts:28`). 배열엔 ToolSearch만 남는다.
2. 등록/삭제는 시스템 프롬프트 재작성이 아니라 **대화 꼬리에 append되는 `<system-reminder>` 델타 메시지(이름만)**로 알린다.
3. 로드 상태를 저장하는 DB가 없다 — **대화 이력에 남은 델타 메시지 자체가 상태**다.
   매번 이력을 스캔해 "이미 고지한 이름 집합"을 재구성하고, 차집합만 새로 흘린다 (`getDeferredToolsDelta`, `toolSearch.ts:646-706`).

이 노트북에서 하는 것:
1. 가짜 MCP 서버 4개 + 연결/해제 + 델타 고지 파이프라인 구현
2. 세션 중간에 서버를 붙이고 떼도 캐시 HIT가 유지되는 것을 `cached_tokens`로 확인

근거 문서: `cc_agent_bible/toolsearch-생애주기-소스분석.md`, `md_group/mcp-캐시-3전선.md`,
`md_group/클로드코드-mcp-캐싱.md`, `md_group/도구-정렬-캐시보존.md`

## 배경 — MCP 한 서버가 입력에 남기는 흔적 두 곳

프롬프트 캐시 규칙은 앞 노트북과 같습니다: 요청 앞부분이 이전 요청과 **완전히 같아야** 재사용,
무효화는 앞→뒤 단방향. 요청은 `[tools 배열][시스템 지시문][대화 내용]` 순서로 만들어집니다.

MCP 서버 하나는 입력에 흔적을 두 곳 남기는데, 클로드코드는 둘 다 "안정 프리픽스 뒤"로 밀어냅니다:

| 흔적 | 순진한 배치 | 클로드코드의 배치 |
|---|---|---|
| 도구 스키마 | tools 배열 (맨 앞) | tools 배열에서 제외(defer) → 검색 결과(대화)로 전달 |
| 서버 사용 지침 | 시스템 프롬프트 본문 | 동적 경계 뒤 / `mcp_instructions_delta` 어태치먼트 |

세션 중간에 서버가 붙거나 떨어질 때:
- 순진한 설계: tools 배열이 바뀜 → 프리픽스 최전방이 바뀜 → **뒤 전체 캐시 무효화**
- 클로드코드: 대화 꼬리에 이름만 담은 델타 한 장 append → 프리픽스 바이트 불변 → **캐시 유지**

델타 원문 (`messages.ts:4178-4193`):
- 등록: "The following deferred tools are now available via ToolSearch: ..."
- 해제: "The following deferred tools are no longer available (their MCP server disconnected).
  Do not search for them — ToolSearch will return no match: ..."

In [1]:
import json
import re
import time

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-5-nano"


## 1. 가짜 MCP 서버 4개

클로드코드에서 MCP 도구 이름은 `mcp__서버이름__도구이름` 형식입니다.
이 프리픽스가 뒤에서 두 가지 특례를 만듭니다:
- `isMcp: true` → tools 배열에서 **무조건 defer** (`MCPTool.ts:28` — "workflow-specific")
- 검색 시 이름 가중치 상향(12/6) + `mcp__서버이름` **프리픽스 매칭** (그 서버 도구 몽땅)

In [2]:
# MCP 서버별 도구 정의는 공용 모듈 cc_deferred_tools.py로 옮겼다 (툴서치 노트북과 한 소스를 공유).
# 이 노트북은 '연결된 서버의 도구만' 검색에 노출하므로 서버 그룹(MCP_SERVERS)을 그대로 가져온다.
from cc_deferred_tools import make_tool, MCP_SERVERS, ALL_MCP_TOOLS  # make_tool은 확장용으로 함께 노출

print(f"서버 {len(MCP_SERVERS)}개, 도구 {len(ALL_MCP_TOOLS)}개")
for s, tools in MCP_SERVERS.items():
    print(f"  {s}: {', '.join(t['name'] for t in tools)}")


서버 4개, 도구 11개
  slack: mcp__slack__send_message, mcp__slack__read_channel, mcp__slack__search_messages
  github: mcp__github__create_pr, mcp__github__list_issues, mcp__github__merge_pr
  figma: mcp__figma__search_files, mcp__figma__get_design, mcp__figma__export_asset
  supabase: mcp__supabase__run_sql, mcp__supabase__list_tables


## 2. tools 배열은 2개로 동결, 연결 상태는 클라이언트에만

서버가 몇 개 붙든 API에 선언하는 tools는 `tool_search`, `tool_invoke` 2개에서 변하지 않습니다.
`mcp_connect` / `mcp_disconnect`는 클라이언트 상태(`CONNECTED`)만 바꿉니다 —
모델에게 알리는 일은 다음 수집 지점의 **델타 고지**(§3)가 맡습니다.

한 가지 중요한 점: 원본 클로드코드는 "검색 먼저, 실행은 그 다음" 같은 규칙을 시스템 프롬프트에 쓰지 않습니다.
그 지식이 사는 곳은 **ToolSearch 도구의 description**입니다 — deferred 도구는 system-reminder에 이름만 나오고,
스키마를 로드해야 호출할 수 있으며, 쿼리 형식은 select:/키워드라는 설명 전부가 도구 정의 안에 있습니다.
아래 `TOOL_SEARCH_DEF`의 description이 그 구조의 이식입니다. 도구 정의는 동결된 프리픽스의 일부라 캐시 비용이 0입니다.

In [3]:
# 도구 사용법 지식은 시스템 프롬프트가 아니라 여기(도구 description)에 산다 —
# 클로드코드 ToolSearch 도구의 실제 description 구조를 한국어로 이식.
# 주의: MCP 전용이 아니다 — deferred된 모든 도구를 다루는 범용 검색기 (MCP는 항상 defer되는 부류일 뿐)
TOOL_SEARCH_DEF = {
    "type": "function",
    "name": "tool_search",
    "description": (
        "사용 가능한 도구를 검색해 스키마를 로드한다. "
        "일부 도구는 tools에 미리 선언되지 않고(deferred), 대화 중간의 <system-reminder> 고지에 이름만 나타난다. "
        "스키마를 로드하기 전에는 이름만 알 뿐 파라미터를 모르므로 실행할 수 없다 — "
        "먼저 이 도구로 스키마를 받은 다음에만 tool_invoke로 실행할 수 있다. "
        "'no longer available' 고지에 나온 도구는 검색해도 no match를 돌려준다. "
        "쿼리 형식: "
        "(1) 'select:이름1,이름2' — 정확한 이름 직조회, 쉼표로 여러 개. "
        "(2) 일반 키워드 — 조사를 뺀 명사를 공백으로 구분해 후보 검색. 예: '슬랙 메시지 전송'. "
        "(3) '+키워드' — 반드시 매칭돼야 하는 필수 키워드."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "'select:도구이름' 직조회 또는 키워드 검색어",
            }
        },
        "required": ["query"],
    },
    "strict": False,
}

TOOL_INVOKE_DEF = {
    "type": "function",
    "name": "tool_invoke",
    "description": ("tool_search로 스키마를 확인한 도구를 실제로 실행한다. 모든 도구 실행은 이 통로로만 한다. "
                    "스키마를 로드하지 않은 도구를 호출하면 에러와 함께 로드 방법이 안내된다."),
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "실행할 도구 이름"},
            "arguments": {"type": "object", "description": "그 도구의 스키마에 맞춘 인자 객체"},
        },
        "required": ["name", "arguments"],
    },
    "strict": False,
}

FROZEN_TOOLS = [TOOL_SEARCH_DEF, TOOL_INVOKE_DEF]  # 절대 변경하지 않는 배열

CONNECTED = {}  # 서버이름 → True. 연결 상태는 여기(클라이언트)에만 있다


def connected_registry():
    """현재 연결된 서버들의 도구만 모은 뷰. tool_search / tool_invoke는 항상 이것만 본다."""
    return {t["name"]: t for s in CONNECTED for t in MCP_SERVERS[s]}


def mcp_connect(server):
    CONNECTED[server] = True
    print(f"🔌 MCP 서버 '{server}' 연결 — 도구 {len(MCP_SERVERS[server])}개")


def mcp_disconnect(server):
    CONNECTED.pop(server, None)
    print(f"🔌 MCP 서버 '{server}' 연결 해제")

## 3. 델타 고지 — 대화 이력이 곧 상태

클로드코드 `getDeferredToolsDelta`(`toolSearch.ts:646-706`)의 핵심은 **로드 상태를 저장하는 DB가 없다**는 점입니다:

1. 매번 대화 이력을 스캔해서 과거 델타 메시지들의 added/removed를 순서대로 재생 →
   "이미 고지한 이름 집합"을 재구성한다 (`:655-663`)
2. 현재 레지스트리와의 **차집합만** 새 `<system-reminder>`로 대화 꼬리에 append한다
3. 차이가 없으면 아무것도 안 보낸다 (`:677` → null 리턴, 재전송 없음)

상태가 이력 안에 있으니 resume·compact를 넘어도 복원되고, 무엇보다 **append-only**라서
프리픽스가 바이트 단위로 불변 — 캐시가 안 깨집니다. 고지에는 **이름만** 실립니다. 스키마는 여전히 검색으로.

In [4]:
ADDED_HEADER = "The following deferred tools are now available via ToolSearch"
REMOVED_HEADER = "The following deferred tools are no longer available (their MCP server disconnected)"
NAME_RE = re.compile(r"^mcp__[A-Za-z0-9_]+$")


def render_delta(added, removed):
    # 클로드코드 messages.ts:4178-4193 렌더링 문구 보존
    parts = []
    if added:
        parts.append(ADDED_HEADER + ". Their schemas are NOT loaded — "
                     "call tool_search with query \"select:<name>\" before use:\n" + "\n".join(added))
    if removed:
        parts.append(REMOVED_HEADER + ". Do not search for them — tool_search will return no match:\n"
                     + "\n".join(removed))
    return "<system-reminder>\n" + "\n\n".join(parts) + "\n</system-reminder>"


def announced_names(input_list):
    """이력의 델타 메시지들을 순서대로 재생해 '이미 고지한 이름 집합'을 재구성 (toolSearch.ts:655-663 이식)."""
    announced = set()
    for item in input_list:
        if not (isinstance(item, dict) and item.get("role") == "user"):
            continue  # 모델 출력(reasoning, function_call 등)은 dict가 아니라서 자동으로 걸러진다
        text = item.get("content", "")
        if not isinstance(text, str) or "<system-reminder>" not in text:
            continue
        mode = None
        for line in text.splitlines():
            line = line.strip()
            if line.startswith(ADDED_HEADER):
                mode = "add"
            elif line.startswith(REMOVED_HEADER):
                mode = "remove"
            elif NAME_RE.match(line):
                if mode == "add":
                    announced.add(line)
                elif mode == "remove":
                    announced.discard(line)
    return announced


def flush_deferred_delta(sess):
    """수집 지점에서 호출. 차집합만 대화 꼬리에 append, 차이 없으면 no-op (toolSearch.ts:646-706 이식)."""
    current = set(connected_registry())
    announced = announced_names(sess["input_list"])
    added = sorted(current - announced)
    removed = sorted(announced - current)
    if not added and not removed:
        return
    sess["input_list"].append({"role": "user", "content": render_delta(added, removed)})
    if added:
        print(f"    📎 델타 고지(등록): {', '.join(added)}")
    if removed:
        print(f"    📎 델타 고지(해제): {', '.join(removed)}")

## 4. tool_search — MCP 가중치와 mcp__ 프리픽스 매칭

앞 노트북의 검색기를 그대로 쓰되, 클로드코드의 MCP 특례를 추가합니다 (`searchToolsWithKeywords`):
- 이름 단어 가중치: MCP 도구는 정확일치 **12** / 부분포함 **6** (일반 도구는 10/5)
- `select:mcp__figma`처럼 서버 프리픽스로 조회하면 그 서버 도구 **전부**의 스키마를 돌려준다
- 매치 0개면 클로드코드 원문대로 "No matching deferred tools found" —
  해제된 서버의 도구를 검색할 때 이 경로를 탄다

검색 결과(스키마)는 tools 배열이 아니라 **tool_result 본문(대화 꼬리)**으로 흘립니다.
클로드코드 1P는 `tool_reference` 블록(이름만)을 남기고 API 서버가 `<functions>`로 확장하지만
(`toolSearch.ts:569-571`), OpenAI엔 그 서버 확장이 없어서 스키마 텍스트를 직접 담는 방식
(Bedrock/Vertex 대체 경로와 동일)을 씁니다.

In [5]:
TOP_N = 5
DOMINANT_RATIO = 2.0  # 1위 점수가 2위의 2배 이상이면 바로 스키마 리턴


def name_parts(name):
    # 클로드코드 parseToolName 이식: snake_case와 CamelCase를 단어로 분해
    spaced = re.sub(r"([a-z])([A-Z])", r"\1 \2", name).replace("_", " ")
    return [p for p in spaced.lower().split() if p]


def term_matches_text(term, text):
    # 한글이 든 키워드는 substring, 영문은 단어 경계(\b) 매칭
    if re.search(r"[가-힣]", term):
        return term in text
    return re.search(r"\b" + re.escape(term) + r"\b", text) is not None


def score_tool(terms, tool):
    # 클로드코드 searchToolsWithKeywords 스코어링 이식 — MCP 도구는 이름 가중치가 12/6으로 더 높다
    is_mcp = tool["name"].startswith("mcp__")
    w_exact, w_part = (12, 6) if is_mcp else (10, 5)
    parts = name_parts(tool["name"])
    full = " ".join(parts)
    desc = tool["description"].lower()
    hint = tool.get("search_hint", "").lower()
    score = 0
    for term in terms:
        if term in parts:
            score += w_exact     # 이름 단어 정확 일치
        elif any(term in p for p in parts):
            score += w_part      # 이름 단어 부분 포함
        if score == 0 and term in full:
            score += 3           # 이름 전체 문자열 (보조)
        if hint and term_matches_text(term, hint):
            score += 4           # 큐레이션 검색 힌트
        if term_matches_text(term, desc):
            score += 2           # 설명
    return score


def full_schema_text(names):
    blocks = [
        json.dumps(
            {"name": ALL_MCP_TOOLS[n]["name"],
             "description": ALL_MCP_TOOLS[n]["description"],
             "parameters": ALL_MCP_TOOLS[n]["parameters"]},
            ensure_ascii=False, indent=2)
        for n in names
    ]
    return ("도구 스키마:\n" + "\n".join(blocks)
            + "\n\n이제 tool_invoke(name=도구이름, arguments=스키마에 맞는 인자 객체)로 실행하세요.")


def log_search_event(sess, summary):
    if sess is not None:
        sess.setdefault("search_events", []).append(summary)


def deliver_schemas(names, sess, prefix_note=""):
    """스키마를 대화로 흘리면서 세션의 발견 집합(discovered)에 기록 — tool_invoke의 로드 게이트가 이걸 본다.
    클로드코드의 extractDiscoveredToolNames가 tool_result의 tool_reference를 스캔해 만드는 집합에 해당."""
    if sess is not None:
        sess.setdefault("discovered", set()).update(names)
    log_search_event(sess, "스키마:" + ",".join(names))
    return prefix_note + full_schema_text(names)


def handle_tool_search(query, sess=None):
    query = query.strip()
    registry = connected_registry()  # 연결된 서버의 도구만 보인다

    # 모드 A — select: 정확한 이름 조회 (mcp__서버 프리픽스면 그 서버 몽땅 — CC 프리픽스 매칭)
    if query.lower().startswith("select:"):
        names = [n.strip() for n in query[len("select:"):].split(",") if n.strip()]
        found, missing = [], []
        for n in names:
            if n in registry:
                found.append(n)
            else:
                prefix_hits = [k for k in registry if k.startswith(n + "__")]
                if prefix_hits:
                    found += prefix_hits
                else:
                    missing.append(n)
        if not found:
            log_search_event(sess, "결과없음")
            return ("No matching deferred tools found.\n"
                    f"없는 이름: {', '.join(missing)}. "
                    f"현재 연결된 MCP 서버: {', '.join(CONNECTED) or '없음'}. "
                    "해당 서버가 연결 해제됐다면 검색을 반복하지 말고 사용자에게 알리세요.")
        note = f"\n\n(없는 이름이라 제외됨: {', '.join(missing)})" if missing else ""
        return deliver_schemas(found, sess) + note

    q = query.lower()

    # fast path — 쿼리 전체가 도구 이름이면 즉시 스키마, mcp__ 프리픽스면 그 서버 몽땅
    if q in registry:
        return deliver_schemas([q], sess)
    if q.startswith("mcp__"):
        prefix_hits = [k for k in registry if k.startswith(q)]
        if prefix_hits:
            return deliver_schemas(prefix_hits, sess)

    # 모드 B — 키워드 검색. "+키워드"는 필수 조건
    raw_terms = [t for t in re.split(r"\s+", q) if t]
    required = [t[1:] for t in raw_terms if t.startswith("+") and len(t) > 1]
    optional = [t for t in raw_terms if not t.startswith("+")]
    terms = required + optional if required else raw_terms

    candidates = list(registry.values())
    if required:
        candidates = [t for t in candidates
                      if all(score_tool([r], t) > 0 for r in required)]

    scored = sorted(((score_tool(terms, t), t) for t in candidates), key=lambda x: -x[0])
    scored = [(s, t) for s, t in scored if s > 0]
    if not scored:
        log_search_event(sess, "결과없음")
        return ("No matching deferred tools found.\n"
                f"현재 연결된 MCP 서버: {', '.join(CONNECTED) or '없음'}. "
                "조사를 뺀 다른 키워드로 다시 검색하거나, "
                "해당 서버가 연결 해제됐다면 검색을 반복하지 말고 사용자에게 알리세요.")

    top = scored[:TOP_N]
    if len(top) == 1 or top[0][0] >= DOMINANT_RATIO * top[1][0]:
        name = top[0][1]["name"]
        return deliver_schemas([name], sess, "1위 점수가 압도적이라 바로 스키마를 리턴합니다.\n\n")

    log_search_event(sess, "후보:" + ",".join(t["name"] for _, t in top))
    cards = "\n".join(f"{i + 1}. {t['name']} — {t['description']} (점수 {s})"
                      for i, (s, t) in enumerate(top))
    return ("후보 도구 목록 (점수순):\n" + cards
            + "\n\n필요한 도구를 모두 골라 tool_search(query=\"select:이름1,이름2\")로 다시 호출하세요."
            + "\n맞는 것이 없으면 다른 키워드로 재검색하세요.")

## 5. tool_invoke — 규칙 대신 반응형 에러 힌트

원본 클로드코드는 "검색 먼저"를 시스템 프롬프트 규칙으로 강제하지 않습니다.
deferred 도구를 스키마 없이 호출하면 **그 순간** 하네스가 힌트를 돌려줍니다
(`buildSchemaNotSentHint`, `toolExecution.ts:578-598` —
"Load the tool first: call ToolSearch with query \"select:{도구명}\", then retry this call.").

같은 구조로 실행 게이트웨이에 반응형 힌트 두 개를 둡니다:
- **스키마 미로드 호출** → "먼저 tool_search(select:...)로 로드하고 재시도하라" (위 힌트의 이식)
- **해제된 서버의 도구 호출** → "서버가 해제됐다" → 모델이 재시도 대신 사용자에게 보고

In [6]:
TYPE_CHECK = {"string": str, "integer": int, "number": (int, float),
              "boolean": bool, "array": list, "object": dict}


def validate_args(parameters, args):
    errors = []
    props = parameters.get("properties", {})
    for required in parameters.get("required", []):
        if required not in args:
            errors.append(f"필수 인자 '{required}' 누락")
    for key, value in args.items():
        if key not in props:
            errors.append(f"스키마에 없는 인자 '{key}'")
        elif not isinstance(value, TYPE_CHECK.get(props[key]["type"], object)):
            errors.append(f"'{key}'는 {props[key]['type']} 타입이어야 함")
    return errors


def handle_tool_invoke(name, arguments, sess=None):
    registry = connected_registry()
    tool = registry.get(name)
    if tool is None:
        if name in ALL_MCP_TOOLS:
            server = name.split("__")[1]
            return (f"ERROR: '{name}'의 MCP 서버 '{server}'가 연결 해제되어 실행할 수 없습니다. "
                    "재시도하지 말고 사용자에게 알리세요.")
        return f"ERROR: '{name}' 도구는 없습니다. tool_search로 먼저 조회하세요."
    if sess is not None and name not in sess.get("discovered", set()):
        # 클로드코드 buildSchemaNotSentHint(toolExecution.ts:578-598) 이식 — 프롬프트 규칙 대신 반응형 힌트
        return (f"ERROR: '{name}'의 스키마가 아직 로드되지 않았습니다. "
                f"먼저 tool_search(query=\"select:{name}\")로 스키마를 로드한 뒤 이 호출을 다시 시도하세요.")
    if isinstance(arguments, str):  # 모델이 객체 대신 JSON 문자열로 보낸 경우
        try:
            arguments = json.loads(arguments)
        except json.JSONDecodeError:
            return "ERROR: arguments가 올바른 JSON 객체가 아닙니다. 스키마에 맞춰 다시 호출하세요."
    errors = validate_args(tool["parameters"], arguments)
    if errors:
        return "ERROR: 인자 검증 실패 — " + "; ".join(errors) + ". 스키마에 맞춰 다시 호출하세요."
    return tool["handler"](arguments)

## 6. API 호출 없이 동작 먼저 확인

델타 파이프라인부터: 연결 → 첫 고지(전체), 등록/해제 → 차집합만, 변화 없음 → no-op.

In [7]:
CONNECTED.clear()
mcp_connect("slack")
mcp_connect("github")

fake = {"input_list": []}
flush_deferred_delta(fake)  # 첫 수집 지점 — 연결된 6개 전부 고지
print("\n─── 첫 고지 메시지 ───")
print(fake["input_list"][-1]["content"])

mcp_connect("figma")
mcp_disconnect("github")
flush_deferred_delta(fake)  # 차집합만 — figma 3개 등록 + github 3개 해제
print("\n─── 두 번째 고지 메시지 (차집합만) ───")
print(fake["input_list"][-1]["content"])

print("\n이력 스캔으로 재구성한 고지 집합:")
print(sorted(announced_names(fake["input_list"])))

before = len(fake["input_list"])
flush_deferred_delta(fake)
print("\n변화 없이 다시 flush →", "append 0건 (no-op)" if len(fake["input_list"]) == before else "append 발생?!")

🔌 MCP 서버 'slack' 연결 — 도구 3개
🔌 MCP 서버 'github' 연결 — 도구 3개
    📎 델타 고지(등록): mcp__github__create_pr, mcp__github__list_issues, mcp__github__merge_pr, mcp__slack__read_channel, mcp__slack__search_messages, mcp__slack__send_message

─── 첫 고지 메시지 ───
<system-reminder>
The following deferred tools are now available via ToolSearch. Their schemas are NOT loaded — call tool_search with query "select:<name>" before use:
mcp__github__create_pr
mcp__github__list_issues
mcp__github__merge_pr
mcp__slack__read_channel
mcp__slack__search_messages
mcp__slack__send_message
</system-reminder>
🔌 MCP 서버 'figma' 연결 — 도구 3개
🔌 MCP 서버 'github' 연결 해제
    📎 델타 고지(등록): mcp__figma__export_asset, mcp__figma__get_design, mcp__figma__search_files
    📎 델타 고지(해제): mcp__github__create_pr, mcp__github__list_issues, mcp__github__merge_pr

─── 두 번째 고지 메시지 (차집합만) ───
<system-reminder>
The following deferred tools are now available via ToolSearch. Their schemas are NOT loaded — call tool_search with query "select:<name>" bef

In [8]:
# 위 셀 끝 상태: slack + figma 연결, github 해제
demo = {"input_list": [], "search_events": [], "discovered": set()}

print("─── 스키마 로드 없이 실행 → 반응형 힌트 (buildSchemaNotSentHint 이식) ───")
print(handle_tool_invoke("mcp__slack__send_message", {"channel": "#dev", "text": "hi"}, demo), "\n")

print("─── 키워드 검색 (힌트 매칭) ───")
print(handle_tool_search("피그마 디자인 파일 검색", demo), "\n")

print("─── 서버 프리픽스 select → 그 서버 도구 몽땅 ───")
print(handle_tool_search("select:mcp__figma", demo)[:400], "…\n")

print("─── 스키마 로드 후 같은 실행 → 통과 ───")
print(handle_tool_invoke("mcp__figma__search_files", {"keyword": "로그인"}, demo), "\n")

print("─── 해제된 서버의 도구 검색 → no match ───")
print(handle_tool_search("깃허브 이슈", demo), "\n")

print("─── 해제된 서버의 도구 실행 → 거부 ───")
print(handle_tool_invoke("mcp__github__list_issues", {"repo": "acme/web"}, demo))

─── 스키마 로드 없이 실행 → 반응형 힌트 (buildSchemaNotSentHint 이식) ───
ERROR: 'mcp__slack__send_message'의 스키마가 아직 로드되지 않았습니다. 먼저 tool_search(query="select:mcp__slack__send_message")로 스키마를 로드한 뒤 이 호출을 다시 시도하세요. 

─── 키워드 검색 (힌트 매칭) ───
1위 점수가 압도적이라 바로 스키마를 리턴합니다.

도구 스키마:
{
  "name": "mcp__figma__search_files",
  "description": "피그마에서 디자인 파일을 검색할 때 사용.",
  "parameters": {
    "type": "object",
    "properties": {
      "keyword": {
        "type": "string",
        "description": "검색 키워드"
      }
    },
    "required": [
      "keyword"
    ]
  }
}

이제 tool_invoke(name=도구이름, arguments=스키마에 맞는 인자 객체)로 실행하세요. 

─── 서버 프리픽스 select → 그 서버 도구 몽땅 ───
도구 스키마:
{
  "name": "mcp__figma__search_files",
  "description": "피그마에서 디자인 파일을 검색할 때 사용.",
  "parameters": {
    "type": "object",
    "properties": {
      "keyword": {
        "type": "string",
        "description": "검색 키워드"
      }
    },
    "required": [
      "keyword"
    ]
  }
}
{
  "name": "mcp__figma__get_design",
  "description": "피그마 디자인 파일의 상세 

## 7. 에이전트 루프와 캐시 측정 — 턴 시작이 수집 지점

클로드코드에서 델타가 실리는 타이밍은 푸시가 아니라 **풀(pull)**입니다. 서버가 붙는 순간이 아니라:
- 케이스 A: 유휴 상태면 다음 유저 엔터 → **턴 시작 수집** (이 노트북이 재현하는 경로)
- 케이스 B: 턴 진행 중이면 도구 실행 직후 사이클 꼬리에 자동 부착 (`query.ts:1569`)

`run_turn`은 유저 메시지를 넣기 **직전에** `flush_deferred_delta`를 호출합니다.
델타 메시지가 유저 메시지보다 먼저 대화에 실리는 순서까지 클로드코드와 같습니다.

In [9]:
def new_session(tools, instructions, cache_key):
    return {"tools": list(tools), "instructions": instructions, "cache_key": cache_key,
            "input_list": [], "log": [], "turn": 0, "search_events": [], "discovered": set()}


def record_usage(sess, response, elapsed):
    usage = response.usage
    cached = getattr(usage.input_tokens_details, "cached_tokens", 0) or 0
    cycle = sum(1 for r in sess["log"] if r["turn"] == sess["turn"]) + 1  # 턴 내 몇 번째 사이클인지
    row = {"req": len(sess["log"]) + 1, "turn": sess["turn"], "cycle": cycle, "input": usage.input_tokens,
           "cached": cached, "output": usage.output_tokens,
           "search": "-", "sec": round(elapsed, 1)}
    sess["log"].append(row)
    mark = "✅ HIT" if cached > 0 else "❌ MISS"
    print(f"    [요청 {row['req']:>2} · 사이클 {cycle}] input={row['input']:>6}  cached={cached:>6}  {mark}"
          f"  ({row['sec']}초)")


def dispatch(sess, name, args):
    if name == "tool_search":
        return handle_tool_search(args.get("query", ""), sess)
    if name == "tool_invoke":
        return handle_tool_invoke(args.get("name", ""), args.get("arguments", {}), sess)
    return f"ERROR: '{name}'은 직접 호출할 수 없습니다. tool_invoke를 사용하세요."


def run_turn(sess, user_msg, max_requests=8):
    sess["turn"] += 1
    print(f"\n👤 사용자: {user_msg}")
    flush_deferred_delta(sess)  # 수집 지점 — 델타가 유저 메시지보다 먼저 실린다
    sess["input_list"].append({"role": "user", "content": user_msg})

    for _ in range(max_requests):
        start = time.perf_counter()
        response = client.responses.create(
            model=MODEL,
            instructions=sess["instructions"],
            input=sess["input_list"],
            tools=sess["tools"],
            prompt_cache_key=sess["cache_key"],
        )
        record_usage(sess, response, time.perf_counter() - start)
        sess["input_list"] += response.output  # reasoning, function_call 등을 그대로 누적

        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            print(f"🤖 답변: {response.output_text}")
            return

        invoked = []
        for call in calls:
            args = json.loads(call.arguments)
            result = dispatch(sess, call.name, args)
            if call.name == "tool_invoke":
                invoked.append("실행:" + str(args.get("name", "?")))
            preview = call.arguments if len(call.arguments) <= 90 else call.arguments[:90] + "…"
            print(f"    🔧 {call.name}({preview})")
            print(f"       → {result.splitlines()[0][:90]}")
            sess["input_list"].append(
                {"type": "function_call_output", "call_id": call.call_id, "output": result})

        events = sess.get("search_events", []) + invoked
        sess["search_events"] = []
        if events:
            sess["log"][-1]["search"] = " · ".join(events)

    print("⚠️ 최대 요청 횟수 도달 — 턴 종료")


def print_log(title, sess):
    print(f"═══ {title} ═══")
    print(f"{'요청':>3} {'턴':>3} {'사이클':>3} {'input':>7} {'cached':>7} {'적중률':>5} {'초':>6}  search_result")
    for r in sess["log"]:
        rate = f"{r['cached'] / r['input'] * 100:.0f}%" if r["input"] else "-"
        sr = r.get("search", "-")
        if len(sr) > 52:
            sr = sr[:52] + "…"
        print(f"{r['req']:>4} {r['turn']:>3} {r['cycle']:>5} {r['input']:>7} {r['cached']:>7} {rate:>6} {r['sec']:>6}  {sr}")
    total_input = sum(r["input"] for r in sess["log"])
    total_cached = sum(r["cached"] for r in sess["log"])
    misses = sum(1 for r in sess["log"] if r["cached"] == 0)
    print(f"합계: 요청 {len(sess['log'])}회 | 입력 {total_input:,} 토큰 | "
          f"캐시에서 재사용 {total_cached:,} 토큰 ({total_cached / total_input * 100:.0f}%) | 미스 {misses}회")

## 8. 시스템 지시문 — 도구 사용법은 여기 안 쓴다

원본 클로드코드의 시스템 프롬프트에는 "ToolSearch로 먼저 검색하고 실행하라" 같은 규칙 블록이 없습니다.
그 지식은 세 군데에 나뉘어 살고, 이 노트북도 같은 위치에 뒀습니다:

1. **tool_search 도구의 description** — deferred/스키마 로드/쿼리 형식 설명 (§2)
2. **델타 고지 문구 자체** — "Their schemas are NOT loaded — call tool_search ... before use" (§3)
3. **반응형 에러 힌트** — 스키마 없이 호출하면 그때 가르쳐준다 (§5, `buildSchemaNotSentHint`)

그래서 시스템 지시문에는 페르소나 한 줄과 도구 사용법과 무관한 일반 업무 정책만 남습니다.
(`[작업 정책]`은 앞 노트북과 동일 — 프리픽스를 캐시 최소 단위(1024토큰) 위로 올리는 역할도 겸함)

In [10]:
PERSONA = "너는 사내 업무 비서다. 사용자의 요청을 도구를 사용해서 처리한다.\n"

COMMON_POLICY = """[작업 정책]
- 날짜는 YYYY-MM-DD 형식, 시각은 HH:MM 24시간 형식으로 도구에 넘긴다. 연도가 없으면 2026년으로 본다.
- 메시지나 메일 본문은 사용자가 준 문구를 그대로 쓰고, 내용을 마음대로 추가하지 않는다.
- 삭제, 머지, 결제처럼 되돌리기 어려운 작업은 실행하기 전에 사용자에게 한 번 확인한다.
- 개인정보는 도구 인자에 꼭 필요한 경우에만 넣는다.
- 검색 결과가 비어 있으면 키워드를 바꿔서 한 번 더 검색하고, 그래도 없으면 없다고 보고한다.
- 한 요청에 여러 작업이 있으면 순서대로 하나씩 처리한다.
- 도구 실행 결과에 오류가 있으면 그 내용을 사용자에게 그대로 알린다.
- 실행하지 않은 작업을 했다고 말하지 않는다.
- 필요한 인자가 요청에 없으면 합리적인 값을 채우되, 최종 답변에서 그 사실을 밝힌다.
- 모든 시각은 한국 표준시(Asia/Seoul) 기준으로 해석한다.
- 도구를 실행하기 전에 스키마의 required 목록에 있는 인자가 전부 채워졌는지 확인한다.
- 같은 도구를 같은 인자로 두 번 연속 호출하지 않는다.
- 사용자가 시키지 않은 도구 실행은 하지 않는다.
- 도구 이름과 인자 이름은 스키마에 적힌 그대로 쓴다. 임의로 줄이거나 바꾸지 않는다.
- 답변 첫머리에 인사말, 감탄사, 이모지를 넣지 않는다.
- 작업이 끝나면 어떤 도구를 실행했고 결과가 무엇이었는지 한 줄로 요약해서 답한다.
- 최종 답변은 한국어로 간결하게 쓴다.
"""

FORMAT_POLICY = """[응답 형식 정책]
- 도구 실행 전에 어떤 도구를 왜 쓰는지 속으로만 판단하고, 사용자에게는 결과만 보고한다.
- 여러 도구를 실행했으면 실행한 순서대로 결과를 정리한다.
- 숫자, 날짜, 채널 이름, 파일 키 같은 식별자는 도구 결과에 나온 그대로 인용한다.
- 도구 결과가 길면 사용자 질문과 관련된 부분만 추려서 전달한다.
- 같은 턴에서 이미 확인한 정보는 다시 도구를 호출하지 않고 재사용한다.
- 도구가 빈 결과를 돌려주면 결과가 없다는 사실을 그대로 보고한다.
- 추측으로 정보를 만들어내지 않는다. 도구 결과에 없는 내용은 없다고 말한다.
- 작업을 절반만 처리한 경우 어디까지 됐고 무엇이 남았는지 명확히 구분해서 알린다.
- 에러 메시지를 사용자에게 전달할 때는 원문 그대로 인용한 뒤 한 줄 설명을 덧붙인다.
- 목록을 보고할 때는 항목당 한 줄로 정리하고, 항목이 다섯 개를 넘으면 개수를 먼저 말한다.
- 사용자가 요청한 범위를 넘는 정보는 묻기 전에는 덧붙이지 않는다.
"""

SAFETY_POLICY = """[안전 정책]
- 되돌리기 어려운 작업(삭제, 머지, 대량 발송)은 실행 전에 대상과 범위를 한 번 더 확인한다.
- 외부로 나가는 메시지에는 내부 식별자나 토큰 값을 포함하지 않는다.
- 사용자가 명시하지 않은 수신자를 임의로 추가하지 않는다.
- 민감한 값(비밀번호, 키)은 출력에 마스킹해서 표시한다.
- 실패한 작업을 성공했다고 요약하지 않는다.
- 오래 걸리는 작업은 시작했다는 사실을 먼저 알린다.
- 동일 요청이 반복되면 직전 결과를 재사용할지 사용자에게 확인한다.
- 도구 결과와 사용자 기대가 상충하면 도구 결과를 기준으로 보고하고 차이를 명시한다.
"""

# 정책 블록들이 동결 프리픽스를 캐시 최소 단위(1024토큰) 위로 여유 있게 올린다
INSTRUCTIONS = PERSONA + "\n" + COMMON_POLICY + "\n" + FORMAT_POLICY + "\n" + SAFETY_POLICY
print(f"지시문 길이: {len(INSTRUCTIONS)}자 — 도구 사용 규칙 없음")

지시문 길이: 1589자 — 도구 사용 규칙 없음


## 9. 실측 — 서버를 붙이고 떼도 캐시가 산다

시나리오:
1. slack + github 연결 상태로 세션 시작 → 턴 1 (첫 수집 지점에서 전체 고지가 한 번 나간다)
2. 세션 중간에 figma **등록** → 턴 2: 피그마 작업
3. 세션 중간에 github **해제** → 턴 3: 깃허브 작업 요청 (모델이 해제 사실을 보고해야 함)
4. 세션 중간에 supabase **등록** → 턴 4: DB 작업

내내 tools 배열과 지시문은 바이트 불변입니다. 볼 것은 등록/해제 직후 턴의 첫 요청 `cached`.

In [11]:
CONNECTED.clear()
mcp_connect("slack")
mcp_connect("github")

sess = new_session(tools=FROZEN_TOOLS, instructions=INSTRUCTIONS, cache_key="mcp-delta-1")
run_turn(sess, "슬랙 #dev 채널에 '배포 완료' 공지 올려줘")

🔌 MCP 서버 'slack' 연결 — 도구 3개
🔌 MCP 서버 'github' 연결 — 도구 3개

👤 사용자: 슬랙 #dev 채널에 '배포 완료' 공지 올려줘
    📎 델타 고지(등록): mcp__github__create_pr, mcp__github__list_issues, mcp__github__merge_pr, mcp__slack__read_channel, mcp__slack__search_messages, mcp__slack__send_message
    [요청  1 · 사이클 1] input=  1329  cached=     0  ❌ MISS  (9.3초)
    🔧 tool_search({"query":"select:mcp__slack__send_message"})
       → 도구 스키마:
    [요청  2 · 사이클 2] input=  2419  cached=  2176  ✅ HIT  (13.7초)
    🔧 tool_invoke({"name":"mcp__slack__send_message","arguments":{"channel":"#dev","text":"배포 완료"}})
       → [가짜 실행 결과] #dev 채널에 메시지 전송 완료: "배포 완료"
    [요청  3 · 사이클 3] input=  2577  cached=  2432  ✅ HIT  (6.4초)
🤖 답변: 도구 실행: mcp__slack__send_message / 결과: #dev 채널에 '배포 완료' 메시지 전송 완료.


In [12]:
mcp_connect("figma")   # 세션 중간 등록 — tools 배열은 안 바뀐다
run_turn(sess, "피그마에서 '로그인 화면' 디자인 파일 찾아줘")

🔌 MCP 서버 'figma' 연결 — 도구 3개

👤 사용자: 피그마에서 '로그인 화면' 디자인 파일 찾아줘
    📎 델타 고지(등록): mcp__figma__export_asset, mcp__figma__get_design, mcp__figma__search_files
    [요청  4 · 사이클 1] input=  1732  cached=  1280  ✅ HIT  (7.8초)
    🔧 tool_search({"query":"select:mcp__figma__search_files"})
       → 도구 스키마:
    [요청  5 · 사이클 2] input=  2599  cached=  2432  ✅ HIT  (1.5초)
    🔧 tool_invoke({"name":"mcp__figma__search_files","arguments":{"keyword":"로그인 화면"}})
       → [가짜 실행 결과] '로그인 화면' 검색 결과 2건: fig_101 "로그인 화면 v2", fig_087 "로그인 화면 (구버전)"
    [요청  6 · 사이클 3] input=  2684  cached=     0  ❌ MISS  (11.1초)
🤖 답변: 검색 결과:
- fig_101 "로그인 화면 v2"
- fig_087 "로그인 화면 (구버전)"

다음 중 열람/내보내기 원하시는 파일의 ID를 알려주시면 해당 파일을 열람(get_design)하거나 내보내기(export_asset)로 진행하겠습니다.

도구 실행: mcp__figma__search_files — 2건 검색 결과를 반환.


In [13]:
mcp_disconnect("github")   # 세션 중간 삭제
run_turn(sess, "깃허브에서 acme/web 저장소 열린 이슈 목록 확인해줘")

🔌 MCP 서버 'github' 연결 해제

👤 사용자: 깃허브에서 acme/web 저장소 열린 이슈 목록 확인해줘
    📎 델타 고지(해제): mcp__github__create_pr, mcp__github__list_issues, mcp__github__merge_pr
    [요청  7 · 사이클 1] input=  2159  cached=  1664  ✅ HIT  (10.9초)
🤖 답변: 현재 시스템에 연결된 GitHub 이슈 조회 도구(mcp__github__list_issues 등)는 더 이상 이용할 수 없습니다.

대체 방법 제안:
- 1) acme/web 저장소의 열린 이슈 목록을 직접 확인해 주시면 제가 필요한 정보로 정리해 드립니다. 예: https://github.com/acme/web/issues?state=open
- 2) 직접 API로 조회하는 방법을 안내해 드립니다. 아래 명령어를 사용해 오픈 이슈를 조회한 뒤 결과를 복사해 보내주시면 정리해 드립니다.
  - 기본: curl -s https://api.github.com/repos/acme/web/issues?state=open
  - 토큰 필요 시: curl -H "Authorization: token YOUR_TOKEN" -s https://api.github.com/repos/acme/web/issues?state=open
  주의: 토큰은 안전하게 보관하고 이 대화에 노출하지 마세요.
- 3) 도구가 다시 연결되면 바로 조회 실행해 드리겠습니다.

어떤 방식으로 진행할지 선택해 주세요.


In [14]:
mcp_connect("supabase")   # 두 번째 등록
run_turn(sess, "수파베이스 public 스키마에 어떤 테이블이 있는지 보여줘")

🔌 MCP 서버 'supabase' 연결 — 도구 2개

👤 사용자: 수파베이스 public 스키마에 어떤 테이블이 있는지 보여줘
    📎 델타 고지(등록): mcp__supabase__list_tables, mcp__supabase__run_sql
    [요청  8 · 사이클 1] input=  2478  cached=  2048  ✅ HIT  (72.2초)
    🔧 tool_search({"query":"select:mcp__supabase__list_tables"})
       → 도구 스키마:
    [요청  9 · 사이클 2] input=  3213  cached=  2944  ✅ HIT  (1.7초)
    🔧 tool_invoke({"name":"mcp__supabase__list_tables","arguments":{"schema_name":"public"}})
       → [가짜 실행 결과] mcp__supabase__list_tables 실행 완료 — 입력: {"schema_name": "public"}
    [요청 10 · 사이클 3] input=  3316  cached=  3200  ✅ HIT  (6.7초)
    🔧 tool_search({"query":"select:mcp__supabase__run_sql"})
       → 도구 스키마:
    [요청 11 · 사이클 4] input=  4211  cached=  3968  ✅ HIT  (1.4초)
    🔧 tool_invoke({"name":"mcp__supabase__run_sql","arguments":{"sql":"SELECT table_name FROM information_sc…)
       → [가짜 실행 결과] mcp__supabase__run_sql 실행 완료 — 입력: {"sql": "SELECT table_name FROM information_
    [요청 12 · 사이클 5] input=  4306  cached=  4224  ✅ HIT  

In [15]:
print_log("MCP 등록/해제 세션 — 델타 방식", sess)

═══ MCP 등록/해제 세션 — 델타 방식 ═══
 요청   턴 사이클   input  cached   적중률      초  search_result
   1   1     1    1329       0     0%    9.3  스키마:mcp__slack__send_message
   2   1     2    2419    2176    90%   13.7  실행:mcp__slack__send_message
   3   1     3    2577    2432    94%    6.4  -
   4   2     1    1732    1280    74%    7.8  스키마:mcp__figma__search_files
   5   2     2    2599    2432    94%    1.5  실행:mcp__figma__search_files
   6   2     3    2684       0     0%   11.1  -
   7   3     1    2159    1664    77%   10.9  -
   8   4     1    2478    2048    83%   72.2  스키마:mcp__supabase__list_tables
   9   4     2    3213    2944    92%    1.7  실행:mcp__supabase__list_tables
  10   4     3    3316    3200    97%    6.7  스키마:mcp__supabase__run_sql
  11   4     4    4211    3968    94%    1.4  실행:mcp__supabase__run_sql
  12   4     5    4306    4224    98%    8.0  -
합계: 요청 12회 | 입력 33,023 토큰 | 캐시에서 재사용 26,368 토큰 (80%) | 미스 2회


위 표에서 볼 것:
- 요청 1은 MISS — 세션 첫 요청이라 캐시가 아직 없는 자리.
- **등록/해제 직후 턴의 첫 요청이 HIT로 나온다는 것** (무작위 미스에 걸리지 않은 한) —
  tools 배열과 지시문이 바이트 단위로 그대로라서, 델타 리마인더는 캐시 입장에선
  "대화가 좀 늘어난 것"에 불과합니다.
- 중간에 무작위 MISS가 몇 개 섞일 수 있습니다 (OpenAI 캐시는 best-effort — 앞 노트북 부록 1과 같은 현상).
  프리픽스가 진짜 깨진 것과 구분하는 법: 프리픽스 파괴라면 그 뒤 **모든** 요청이 연쇄 MISS여야 하는데,
  여기서는 미스 다음 요청이 도로 HIT로 돌아옵니다. 등록/해제 시점과 미스 위치가 상관관계가 없다는 것이 증거.
- search_result 열: `mcp__figma__*`, `mcp__supabase__*`의 검색·실행이 전부 대화 꼬리에서 일어났고,
  턴 3은 검색 없이(또는 no match 후) 해제 보고로 끝납니다.
- 이 설계가 중요한 이유: 실제 클로드코드에서 MCP 서버는 세션 시작 직후 순차적으로 핸드셰이크가 끝나는
  "늦은 연결(late connect)"이 기본입니다. 서버 변동이 tools 배열이나 시스템 프롬프트를 건드리는 설계였다면
  서버가 하나 붙을 때마다 캐시가 깨졌을 것 (`prompts.ts:509-510` 주석
  "busts the prompt cache on late MCP connect" — 그래서 델타 어태치먼트로 이사했다).

## 정리 — 클로드코드 ↔ 이 노트북 대응표

| 클로드코드 | 이 노트북 |
|---|---|
| `isMcp: true` → 무조건 defer (`MCPTool.ts:28`) | MCP 도구를 tools 배열에 안 넣음 (FROZEN_TOOLS 2개 동결) |
| `deferred_tools_delta` — 이력 스캔 + 차집합 (`toolSearch.ts:646-706`) | `announced_names` + `flush_deferred_delta` |
| `<system-reminder>` isMeta user 메시지 (`messages.ts:4178-4193`) | user 롤 system-reminder append (문구 보존) |
| 수집 지점: 턴 시작 / 도구 실행 직후 (`query.ts:1569`) | `run_turn` 시작 시 flush |
| `tool_reference`(이름만) → API 서버가 `<functions>` 확장 | 스키마 텍스트를 tool_result 본문에 직접 |
| MCP 이름 가중 12/6, `mcp__` 프리픽스 매칭 | `score_tool` / `select:mcp__서버` |
| 사용법 지식: 도구 description + 고지 문구 + `buildSchemaNotSentHint` (시스템 프롬프트 아님) | 같은 3위치 — 지시문엔 페르소나+업무 정책만 |
| 캐시 브레이크포인트 "마지막 빌트인 뒤" (`tools.ts:354-356`) | tools+지시문 동결이 같은 역할 |

재현하지 않은 것:
- 서버 지침(instructions)의 `mcp_instructions_delta` — 원리가 도구 고지와 동일해서 생략
- "고지됐다가 defer가 풀렸지만 풀에 남은 도구는 removed로 보고하지 않는다"는 특례 (`toolSearch.ts:641-644`)
- compact 경계에서의 발견 도구 스냅샷 이월 (`toolSearch.ts:550-560`)